# LangGraph com AgentCore Memory usando Estratégia Episódica

## Introdução

Este notebook demonstra como integrar o Amazon Bedrock AgentCore Memory com **estratégia de memória episódica** em um agente de IA conversacional usando o framework LangGraph. Vamos focar na estratégia episódica que captura sessões completas de conversação, permitindo que o agente recorde episódios específicos de planejamento de refeições e acompanhe como os padrões alimentares evoluem ao longo do tempo.

## Detalhes do Tutorial

| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo do tutorial    | Conversacional de longo prazo                                                   |
| Caso de uso do agente | Assistente de Nutrição com Estratégia de Memória Episódica                    |
| Framework de Agentes | LangGraph                                                                       |
| Modelo LLM          | Anthropic Claude Sonnet 3.7                                                     |
| Componentes do tutorial | AgentCore Memory, Estratégia Episódica, LangGraph Hooks, Episódios baseados em Sessão |
| Complexidade do exemplo | Intermediário                                                                |

Você aprenderá a:
- Criar AgentCore Memory com estratégia de memória episódica
- Implementar hooks pre/post model para armazenamento automático de memória
- Construir um assistente de nutrição que lembra sessões de planejamento de refeições
- Recuperar e refletir sobre conversas anteriores
- Acompanhar padrões alimentares ao longo do tempo

### Contexto do Cenário

Neste exemplo, criaremos um **Assistente de Nutrição** que lembra sessões completas de planejamento de refeições usando estratégia de memória episódica. O agente capturará episódios completos de conversação incluindo discussões sobre receitas, substituições de ingredientes e feedback sobre refeições. Isso permite consultas temporais como "O que eu planejei na semana passada?" e análise de padrões de hábitos alimentares.

## Arquitetura

<div style="text-align:left">
    <img src="architecture_episodic.png" width="65%" />
</div>

### Por que Estratégia de Memória Episódica para Nutrição?

- **Baseada em sessão**: Cada conversa de planejamento de refeição é um episódio
- **Contexto temporal**: Refeições estão vinculadas a horários/ocasiões específicas
- **Aprendizado de padrões**: Acompanhe como as preferências evoluem
- **Recordação rica**: Lembre o contexto completo de recomendações anteriores

### Como a Estratégia de Memória Episódica Funciona

A estratégia episódica é projetada para capturar interações como episódios estruturados e refletir sobre esses episódios para gerar insights significativos. Esta estratégia registra não apenas o que aconteceu, mas também a intenção, os pensamentos e o resultado de cada episódio.

#### Três Passos na Estratégia Episódica:

1. **Extração** – Identifica insights úteis da memória de curto prazo para colocar na memória de longo prazo como registros de memória
2. **Consolidação** – Determina se deve gravar informações úteis em um novo registro ou em um registro existente
3. **Reflexão** – Insights são gerados entre episódios a partir das interações do agente

#### Saída da Estratégia:

**Episódios** (formatados em XML):
- Divididos em: situação, intenção, avaliação, justificativa e reflexão no nível do episódio
- Analisados turno a turno conforme a interação prossegue
- Ajudam a entender a ordem das operações e o uso de ferramentas

**Reflexões** (geradas em segundo plano):
- Consolidam múltiplos episódios
- Extraem insights mais amplos identificando:
  - Estratégias e padrões bem-sucedidos
  - Melhorias potenciais
  - Modos de falha comuns
  - Lições aprendidas abrangendo múltiplas interações

#### Para o Assistente de Nutrição:

- **Episódios**: Cada sessão de planejamento de refeição (receitas discutidas, ingredientes, decisões)
- **Reflexões**: Padrões alimentares, culinárias favoritas, progressão de habilidades culinárias
- **Turno a turno**: Exploração de receitas → perguntas sobre ingredientes → substituições → escolha final

## Pré-requisitos

- Python 3.10+
- Conta AWS com permissões apropriadas
- IAM role AWS com permissões apropriadas para AgentCore Memory
- Acesso aos modelos Amazon Bedrock

Vamos começar configurando nosso ambiente!


In [ ]:
# Install necessary libraries from https://github.com/langchain-ai/langchain-aws
%pip install -qr requirements.txt

In [ ]:
import os
import logging

# Import LangGraph and LangChain components
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
import uuid


region = os.getenv("AWS_REGION", "us-east-1")
logging.getLogger("nutrition-agent").setLevel(logging.DEBUG)

In [ ]:
# Import the AgentCoreMemoryStore that we will use as a store
from langgraph_checkpoint_aws import AgentCoreMemoryStore

# For this example, we will just use an InMemorySaver to save context.
# In production, we highly recommend the AgentCoreMemorySaver as a checkpointer which works seamlessly alongside the memory store
# from langgraph_checkpoint_aws import AgentCoreMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from bedrock_agentcore.memory import MemoryClient

In [ ]:
import boto3
import json

# Create IAM role for memory execution
iam_client = boto3.client("iam")
sts_client = boto3.client("sts")
account_id = sts_client.get_caller_identity()["Account"]

ROLE_NAME = "AgentCoreMemoryExecutionRole"

# Trust policy for AgentCore Memory (gamma endpoints)
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": [
                    "preprod.genesis-service.aws.internal",
                    "bedrock-agentcore.amazonaws.com",
                    "developer.genesis-service.aws.internal",
                ]
            },
            "Action": "sts:AssumeRole",
        }
    ],
}

# Permissions for Bedrock model invocation
permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
            "Resource": [
                "arn:aws:bedrock:*::foundation-model/*",
                "arn:aws:bedrock:*:*:inference-profile/*",
            ],
        }
    ],
}

try:
    # Try to get existing role
    role = iam_client.get_role(RoleName=ROLE_NAME)
    MEMORY_EXECUTION_ROLE_ARN = role["Role"]["Arn"]
    print(f"✅ Using existing role: {MEMORY_EXECUTION_ROLE_ARN}")
except iam_client.exceptions.NoSuchEntityException:
    # Create role
    print(f"Creating IAM role: {ROLE_NAME}")
    role = iam_client.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Execution role for AgentCore Memory with custom strategies",
    )
    MEMORY_EXECUTION_ROLE_ARN = role["Role"]["Arn"]

    # Attach inline policy
    iam_client.put_role_policy(
        RoleName=ROLE_NAME,
        PolicyName="BedrockModelAccess",
        PolicyDocument=json.dumps(permissions_policy),
    )
    print(f"✅ Created role: {MEMORY_EXECUTION_ROLE_ARN}")
    print("⏳ Waiting 10 seconds for IAM propagation...")
    import time

    time.sleep(10)

print(f"\nRole ARN: {MEMORY_EXECUTION_ROLE_ARN}")

In [ ]:
memory_name = "NutritionAssistantEpisodic"
client = MemoryClient(region_name=region)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

override_strategy = {
    "customMemoryStrategy": {
        "name": "NutritionEpisodicExtractor",
        "description": "Nutrition assistant with episodic memory for meal planning insights",
        "namespaceTemplates": ["nutrition/{actorId}/{sessionId}/"],
        "configuration": {
            "episodicOverride": {
                "extraction": {
                    "modelId": MODEL_ID,
                    "appendToPrompt": "Extract meal planning conversations including recipes discussed, ingredients mentioned, dietary considerations, and user feedback.",
                },
                "consolidation": {
                    "modelId": MODEL_ID,
                    "appendToPrompt": "Consolidate meal planning sessions into episodes, capturing the flow of recipe exploration and decision-making.",
                },
                "reflection": {
                    "modelId": MODEL_ID,
                    "appendToPrompt": "Generate insights about dietary patterns, favorite recipes, and how meal preferences evolve over time.",
                    "namespaceTemplates": ["nutrition/{actorId}/"],
                },
            }
        },
    }
}

memory = client.create_or_get_memory(
    name=memory_name,
    description="Nutrition assistant with episodic memory for meal planning sessions",
    memory_execution_role_arn=MEMORY_EXECUTION_ROLE_ARN,
    strategies=[override_strategy],
)
memory_id = memory["id"]

print(f"✅ Created episodic memory: {memory_id}")

### Visão Geral da Configuração de Memória

Nossa configuração de AgentCore Episodic Memory inclui:

- **Extração**: Captura conversas de planejamento de refeições com receitas, ingredientes e feedback
- **Consolidação**: Agrupa conversas em episódios de planejamento de refeições
- **Reflexão**: Gera insights sobre padrões alimentares e preferências ao longo do tempo
- **Namespaces**: Organiza episódios por usuário (`nutrition/{actorId}/`)

Cada sessão de conversa se torna um episódio que pode ser recuperado e analisado.

## Passo 3: Inicializar Memory Store e LLM

Agora vamos inicializar o AgentCore Memory Store e nosso modelo de linguagem.

In [ ]:
# Initialize the store to enable long term memory saving and retrieval
store = AgentCoreMemoryStore(memory_id=memory_id, region_name=region)

# Initialize Bedrock LLM
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

## Passo 4: Implementar Memory Hooks

Criaremos hooks pre e post model para lidar automaticamente com o armazenamento de memória:

- **Hook pre-model**: Salva a mensagem do usuário antes da invocação do LLM
- **Hook post-model**: Salva a resposta do assistente após a invocação do LLM

### Como o Processamento de Memória Funciona

1. As mensagens são salvas no AgentCore Memory com actor_id e session_id
2. A estratégia episódica processa as conversas para criar episódios estruturados
3. Os episódios são armazenados no namespace `nutrition/{actorId}/{sessionId}/` com análise turno a turno
4. Reflexões são geradas entre episódios e armazenadas no namespace `nutrition/{actorId}/`
5. Cada episódio captura situação, intenção, avaliação e fluxo da conversa

**Nota**: Os tipos de mensagem do LangChain são convertidos internamente pelo store para os tipos de mensagem do AgentCore Memory para que possam ser processados adequadamente em episódios e reflexões.


In [ ]:
def pre_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """Hook that runs pre-LLM invocation to save the latest human message"""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]
    # Saving the message to the actor and session combination that we get at runtime
    namespace = (actor_id, thread_id)

    messages = state.get("messages", [])
    # Save the last human message we see before LLM invocation
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break

    # For episodic strategy, we just save messages - no retrieval needed
    # Episodes and reflections are generated automatically in the background
    return {"messages": messages}


def post_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """Hook that runs post-LLM invocation to save the assistant response"""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]

    # Saving the message to the actor and session combination that we get at runtime
    namespace = (actor_id, thread_id)

    messages = state.get("messages", [])
    # Save the LLM's response to AgentCore Memory
    for msg in reversed(messages):
        if isinstance(msg, AIMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break

    return {"messages": messages}

## Passo 5: Criar o Agente LangGraph

Agora criaremos nosso agente assistente de nutrição usando o `create_react_agent` do LangGraph com nossos memory hooks integrados. O tool node conterá apenas nossa ferramenta de recuperação de memória de longo prazo e os hooks pre e post model são especificados como argumentos.

**Nota**: para implementações de agentes personalizados, o Store e as ferramentas podem ser configurados para executar conforme necessário para qualquer workflow seguindo este padrão. Hooks pre/post model podem ser usados, toda a conversa pode ser salva no final, etc.

In [ ]:
graph = create_react_agent(
    llm,
    store=store,
    tools=[],  # No additional tools needed for this example
    checkpointer=InMemorySaver(),  # For conversation state management
    pre_model_hook=pre_model_hook,  # Saves user message before LLM call
    post_model_hook=post_model_hook,  # Saves assistant response for episodic processing after LLM call
)

## Passo 6: Configurar Runtime do Agente

Precisamos configurar o agente com identificadores únicos para o usuário e a sessão. Esses IDs são cruciais para a organização e recuperação de memória.

### Input de Invocação do Graph
Precisamos apenas passar a mensagem mais recente do usuário como argumento `inputs`. Isso poderia incluir outras variáveis de estado também, mas para o simples `create_react_agent`, precisamos apenas de messages.

### LangGraph RuntimeConfig
No LangGraph, config é um `RuntimeConfig` que contém atributos necessários no momento da invocação, por exemplo, IDs de usuário ou IDs de sessão. Para o `AgentCoreMemorySaver`, `thread_id` e `actor_id` devem ser definidos no config. Por exemplo, seu endpoint de invocação do AgentCore poderia atribuir isso com base na identidade ou ID do usuário do chamador. Você pode ler a [documentação adicional aqui](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/)



In [ ]:
actor_id = "user-1"
config = {
    "configurable": {
        "thread_id": "session-1",  # REQUIRED: This maps to Bedrock AgentCore session_id under the hood
        "actor_id": actor_id,  # REQUIRED: This maps to Bedrock AgentCore actor_id under the hood
    }
}

## Passo 7: Testar o Agente

Vamos testar nosso assistente de nutrição tendo uma conversa sobre preferências alimentares. O agente capturará automaticamente a conversa como episódios para futura recuperação e análise de padrões.

In [ ]:
# Helper function to pretty print agent output while running
def run_agent(query: str, config: RunnableConfig):
    printed_ids = set()
    events = graph.stream(
        {"messages": [{"role": "user", "content": query}]},
        config,
        stream_mode="values",
    )
    for event in events:
        if "messages" in event:
            for msg in event["messages"]:
                # Check if we've already printed this message
                if id(msg) not in printed_ids:
                    msg.pretty_print()
                    printed_ids.add(id(msg))


prompt = """
Hey there! Im cooking one of my favorite meals tonight, salmon with rice and veggies (healthy). Has
great macros for my weightlifting competition that is coming up. What can I add to this dish to make it taste better
and also improve the protein and vitamins I get?
"""

run_agent(prompt, config)

### O que foi armazenado?
Como você pode ver, o modelo ainda não possui insights de sessões anteriores de planejamento de refeições.

Para esta implementação com hooks pre/post model, duas mensagens foram armazenadas aqui. A primeira mensagem do usuário e a resposta do modelo de IA foram ambas armazenadas como eventos conversacionais no AgentCore Memory. Pode levar alguns momentos para que os episódios e reflexões sejam gerados, então tente novamente após alguns minutos se nada for encontrado na primeira tentativa.

Essas mensagens foram então processadas pela estratégia episódica para criar episódios e reflexões estruturados na memória de longo prazo do AgentCore. Na verdade, podemos verificar o store nós mesmos para verificar o que foi armazenado até agora:

In [ ]:
# Search our conversation messages
search_namespace = ("nutrition", actor_id, "session-1/")
result = store.search(search_namespace, query="meal", limit=3)
print(f"Conversation messages result: {result}")

In [ ]:
# The correct way to search episodic long-term memories in LangGraph
from bedrock_agentcore.memory import MemoryClient

# Use the memory client directly (not the store)
memory_client = MemoryClient(region_name=region)

print("=== Searching Long-Term Episodic Memories ===")
print(f"Memory ID: {memory_id}")
print()

# Search episodic memories (episodes)
print("1. Episodic namespace: nutrition/user-1/session-1/")
try:
    episodes = memory_client.retrieve_memories(
        memory_id=memory_id,
        namespace="nutrition/user-1/session-1/",
        query="meal",
        top_k=3,
    )
    print(f"   Found {len(episodes)} episode memories")
    for mem in episodes:
        content = mem.get("content", {})
        text = content.get("text", str(content))
        print(f"   - {text[:300]}...")
except Exception as e:
    print(f"   Error: {e}")
print()

# Search reflection memories
print("2. Reflection namespace: nutrition/user-1/")
try:
    reflections = memory_client.retrieve_memories(
        memory_id=memory_id, namespace="nutrition/user-1/", query="meal", top_k=3
    )
    print(f"   Found {len(reflections)} reflection memories")
    for mem in reflections:
        content = mem.get("content", {})
        text = content.get("text", str(content))
        print(f"   - {text[:300]}...")
except Exception as e:
    print(f"   Error: {e}")

### Acesso do agente ao store

**Nota** - como o AgentCore memory processa esses eventos em segundo plano, pode levar alguns minutos para que a memória seja extraída e incorporada à recuperação de memória de longo prazo.

Ótimo! Agora vimos que memórias de longo prazo foram extraídas para nossos namespaces com base nas mensagens anteriores da conversa.

Agora, vamos iniciar uma nova sessão e perguntar sobre recomendações do que cozinhar para o jantar. O agente pode usar o store para acessar as memórias de longo prazo que foram extraídas para fazer uma recomendação que o usuário certamente irá gostar.

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-2",  # New session ID
        "actor_id": actor_id,  # Same actor ID
    }
}

run_agent("Today's a new day, what should I make for dinner tonight?", config)

### Conclusão

Como você pode ver, as conversas do agente são automaticamente capturadas e processadas em episódios estruturados com análise turno a turno. A estratégia episódica gera insights entre múltiplas sessões de planejamento de refeições para identificar padrões e acompanhar como as preferências evoluem ao longo do tempo.

O AgentCoreMemoryStore é muito flexível e pode ser implementado de diversas formas, incluindo hooks pre/post model ou apenas ferramentas com operações de store. Usado em conjunto com o AgentCoreMemorySaver para checkpointing, tanto o estado conversacional completo quanto as reflexões episódicas podem ser combinados para formar um sistema de agente complexo e inteligente.